# Divye FE — LR Stacking on OOF

LR meta-learner on OOF predictions from Divye FE models.

**Base models (OOF files needed):**
- `oof_divye_fe.npy` — catboost_divye_fe (AUC 0.95566)
- `oof_cat_cpu.npy` — catboost_default_cpu (AUC 0.95544) ← need to save from cpu notebook

Pattern from `s6e2-stacking.ipynb`: LR meta was C-insensitive → stable signal.  
Previously: LR on seed-avg OOF → LB 0.95352 (second best overall).  
This upgrades the base with the stronger divye_fe OOF.

**If oof_cat_cpu.npy is missing, falls back to oof_cat_seedavg.npy.**

In [1]:
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)
y = train['heart_disease'].values

print(f'Train: {train.shape}  Test: {test.shape}')

Train: (630000, 15)  Test: (270000, 14)


In [2]:
SUB_DIR = Path('submissions')

# Load OOF predictions
oof_divye = np.load(SUB_DIR / 'oof_divye_fe.npy')
print(f'oof_divye_fe:  AUC={roc_auc_score(y, oof_divye):.5f}')

# Load default CPU OOF if available, otherwise seed-avg OOF
if (SUB_DIR / 'oof_cat_cpu.npy').exists():
    oof_cat = np.load(SUB_DIR / 'oof_cat_cpu.npy')
    cat_label = 'oof_cat_cpu'
elif (SUB_DIR / 'oof_cat_seedavg.npy').exists():
    oof_cat = np.load(SUB_DIR / 'oof_cat_seedavg.npy')
    cat_label = 'oof_cat_seedavg'
else:
    raise FileNotFoundError('No catboost OOF found. Run s6e2-catboost-cpu.ipynb and save OOF first.')
print(f'{cat_label}:  AUC={roc_auc_score(y, oof_cat):.5f}')

# Load test predictions
sub_divye = pd.read_csv(SUB_DIR / 'catboost_divye_fe.csv')['Heart Disease'].values

# Identify best available catboost test preds
cat_test_candidates = ['catboost_default_cpu.csv', 'cat_cpu_seedavg_20seeds.csv']
sub_cat = None
for fname in cat_test_candidates:
    fpath = SUB_DIR / fname
    if fpath.exists():
        sub_cat = pd.read_csv(fpath)['Heart Disease'].values
        print(f'Test preds: {fname}')
        break
if sub_cat is None:
    raise FileNotFoundError('No catboost test predictions found.')

print(f'\nCorrelation between OOFs: {np.corrcoef(oof_divye, oof_cat)[0,1]:.5f}')

oof_divye_fe:  AUC=0.95566
oof_cat_seedavg:  AUC=0.95540
Test preds: catboost_default_cpu.csv

Correlation between OOFs: 0.99868


In [3]:
# Stack: LR meta on [oof_divye, oof_cat] — try C grid
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_meta_train = np.column_stack([oof_divye, oof_cat])
X_meta_test  = np.column_stack([sub_divye,  sub_cat])

for C in [0.01, 0.1, 1.0]:
    lr = LogisticRegression(C=C, max_iter=1000)
    oof_meta = cross_val_predict(lr, X_meta_train, y, cv=SKF, method='predict_proba')[:, 1]
    print(f'C={C:4.2f}  OOF AUC={roc_auc_score(y, oof_meta):.5f}')

# Fit best on full data for test predictions
C_best = 0.1
lr_final = LogisticRegression(C=C_best, max_iter=1000)
lr_final.fit(X_meta_train, y)
test_preds = lr_final.predict_proba(X_meta_test)[:, 1]

print(f'\nCoefficients: divye_fe={lr_final.coef_[0][0]:.4f}  catboost={lr_final.coef_[0][1]:.4f}')

oof_meta_final = cross_val_predict(LogisticRegression(C=C_best, max_iter=1000),
                                   X_meta_train, y, cv=SKF, method='predict_proba')[:, 1]
cv_auc = roc_auc_score(y, oof_meta_final)
print(f'\nStack OOF AUC (C={C_best}): {cv_auc:.5f}')
print(f'divye_fe baseline:          0.95566')
print(f'Delta:                      {cv_auc - 0.95566:+.5f}')

C=0.01  OOF AUC=0.95565
C=0.10  OOF AUC=0.95566
C=1.00  OOF AUC=0.95567

Coefficients: divye_fe=4.1047  catboost=2.4029

Stack OOF AUC (C=0.1): 0.95566
divye_fe baseline:          0.95566
Delta:                      +0.00000


In [4]:
label = 'stack_lr_divye_fe'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
print(f'Saved: {fname}')
print(f'desc:  {desc}')
print(f'OOF AUC: {cv_auc:.5f}  (baseline divye_fe: 0.95566  delta: {cv_auc - 0.95566:+.5f})')

Saved: submissions/stack_lr_divye_fe.csv
desc:  stack_lr_divye_fe | cv_auc=0.9557
OOF AUC: 0.95566  (baseline divye_fe: 0.95566  delta: +0.00000)


In [5]:
import subprocess
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

stack_lr_divye_fe: submitted
